In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import re
import json
import random
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter, defaultdict
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

BASE_DIR = Path("/content/drive/MyDrive/New Jurnal Cross/dataset")
OUT_DIR = Path("/content/drive/MyDrive/New Jurnal Cross/processed_intra_csv")
OUT_DIR.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "emodb_path": BASE_DIR / "emodb",
    "ravdess_path": BASE_DIR / "ravdess",
    "resd_path": BASE_DIR / "resd",

    "output_dir": OUT_DIR,

    "unified_emotions": [
        "angry",
        "disgust",
        "fear",
        "happy",
        "neutral",
        "sad"
    ],

    "seed": SEED
}

print("BASE_DIR:", BASE_DIR)
print("OUT_DIR :", OUT_DIR)

for name, path in [
    ("EmoDB", CONFIG["emodb_path"]),
    ("RAVDESS", CONFIG["ravdess_path"]),
    ("RESD", CONFIG["resd_path"]),
]:
    print(f"{name:8s} exists:", path.exists(), "→", path)

BASE_DIR: /content/drive/MyDrive/New Jurnal Cross/dataset
OUT_DIR : /content/drive/MyDrive/New Jurnal Cross/processed_intra_csv
EmoDB    exists: True → /content/drive/MyDrive/New Jurnal Cross/dataset/emodb
RAVDESS  exists: True → /content/drive/MyDrive/New Jurnal Cross/dataset/ravdess
RESD     exists: True → /content/drive/MyDrive/New Jurnal Cross/dataset/resd


In [3]:
# ============================================================
# Label Mapping
# ============================================================

EMODB_MAP = {
    "W": "angry",
    "E": "disgust",
    "A": "fear",
    "F": "happy",
    "N": "neutral",
    "T": "sad",
    # "L": "boredom" → dibuang
}

RAVDESS_MAP = {
    "01": "neutral",
    # "02": "calm" → dibuang
    "03": "happy",
    "04": "sad",
    "05": "angry",
    "06": "fear",
    "07": "disgust",
    # "08": "surprise" → dibuang
}

RESD_MAP = {
    "angry": "angry",
    "anger": "angry",
    "gnev": "angry",

    "happy": "happy",
    "happiness": "happy",
    "radost": "happy",

    "sad": "sad",
    "sadness": "sad",
    "grust": "sad",

    "fear": "fear",
    "strakh": "fear",

    "disgust": "disgust",
    "otvrashenie": "disgust",

    "neutral": "neutral",
    "neytralnyy": "neutral",

    # "enthusiasm": dibuang
    # "interest": dibuang
}

LABEL_TO_ID = {
    label: idx for idx, label in enumerate(CONFIG["unified_emotions"])
}

ID_TO_LABEL = {
    idx: label for label, idx in LABEL_TO_ID.items()
}

print("LABEL_TO_ID:")
print(LABEL_TO_ID)

LABEL_TO_ID:
{'angry': 0, 'disgust': 1, 'fear': 2, 'happy': 3, 'neutral': 4, 'sad': 5}


In [4]:
def find_wav_files(base_path):
    base_path = Path(base_path)
    return sorted(list(base_path.rglob("*.wav")) + list(base_path.rglob("*.WAV")))


def parse_emodb(base_path):
    """
    EmoDB filename example:
    03a01Fa.wav

    Emotion code is usually at stem[5]:
      W = anger
      L = boredom
      E = disgust
      A = anxiety/fear
      F = happiness
      T = sadness
      N = neutral
    """
    records = []
    files = find_wav_files(base_path)

    for f in files:
        stem = f.stem

        if len(stem) < 6:
            continue

        emotion_code = stem[5].upper()
        emotion = EMODB_MAP.get(emotion_code)

        if emotion is None:
            continue

        speaker = stem[:2]

        records.append({
            "filepath": str(f),
            "filename": f.name,
            "stem": stem,
            "dataset": "emodb",
            "language": "german",
            "speech_style": "monologue",
            "corpus_type": "acted_controlled",
            "emotion_original": emotion,
            "emotion": emotion,
            "label": LABEL_TO_ID[emotion],
            "speaker": speaker,
            "speaker_original": speaker,
            "gender": "unknown",
            "source_split": "all",
            "dialog_context": "monologue",
            "dialog_emoA": emotion,
            "dialog_emoB": emotion,
            "is_augmented": False,
        })

    return pd.DataFrame(records)


df_emodb = parse_emodb(CONFIG["emodb_path"])
print("EmoDB:", df_emodb.shape)
display(df_emodb.head())
print(df_emodb["emotion"].value_counts())

EmoDB: (718, 18)


,filepath,filename,stem,dataset,language,speech_style,corpus_type,emotion_original,emotion,label,speaker,speaker_original,gender,source_split,dialog_context,dialog_emoA,dialog_emoB,is_augmented
0,/content/drive/MyDrive/New Jurnal Cross/datase...,03a01Ab.wav,03a01Ab,emodb,german,monologue,acted_controlled,fear,fear,2,03,03,unknown,all,monologue,fear,fear,False
1,/content/drive/MyDrive/New Jurnal Cross/datase...,03a01Eb.wav,03a01Eb,emodb,german,monologue,acted_controlled,disgust,disgust,1,03,03,unknown,all,monologue,disgust,disgust,False
2,/content/drive/MyDrive/New Jurnal Cross/datase...,03a01Fa.wav,03a01Fa,emodb,german,monologue,acted_controlled,happy,happy,3,03,03,unknown,all,monologue,happy,happy,False
3,/content/drive/MyDrive/New Jurnal Cross/datase...,03a01Nc.wav,03a01Nc,emodb,german,monologue,acted_controlled,neutral,neutral,4,03,03,unknown,all,monologue,neutral,neutral,False
4,/content/drive/MyDrive/New Jurnal Cross/datase...,03a01Tc.wav,03a01Tc,emodb,german,monologue,acted_controlled,sad,sad,5,03,03,unknown,all,monologue,sad,sad,False


emotion
angry      140
sad        125
fear       123
happy      118
disgust    106
neutral    106
Name: count, dtype: int64


In [5]:
def parse_ravdess(base_path):
    """
    RAVDESS filename example:
    03-01-05-01-02-01-12.wav

    Modality   = parts[0]
    Vocal      = parts[1]
    Emotion    = parts[2]
    Intensity  = parts[3]
    Statement  = parts[4]
    Repetition = parts[5]
    Actor      = parts[6]

    Emotion:
      01 neutral
      02 calm      → dibuang
      03 happy
      04 sad
      05 angry
      06 fearful
      07 disgust
      08 surprise  → dibuang

    Actor odd  = male
    Actor even = female
    """
    records = []
    files = find_wav_files(base_path)

    for f in files:
        stem = f.stem
        parts = stem.split("-")

        if len(parts) < 7:
            continue

        emotion_code = parts[2]
        emotion = RAVDESS_MAP.get(emotion_code)

        if emotion is None:
            continue

        actor = parts[6]
        actor_id = int(actor)

        gender = "male" if actor_id % 2 == 1 else "female"

        records.append({
            "filepath": str(f),
            "filename": f.name,
            "stem": stem,
            "dataset": "ravdess",
            "language": "english",
            "speech_style": "monologue",
            "corpus_type": "acted_controlled",
            "emotion_original": emotion,
            "emotion": emotion,
            "label": LABEL_TO_ID[emotion],
            "speaker": actor,
            "speaker_original": actor,
            "gender": gender,
            "source_split": "all",
            "ravdess_modality": parts[0],
            "ravdess_vocal_channel": parts[1],
            "ravdess_emotion_code": parts[2],
            "ravdess_intensity": parts[3],
            "ravdess_statement": parts[4],
            "ravdess_repetition": parts[5],
            "ravdess_actor": parts[6],
            "dialog_context": "monologue",
            "dialog_emoA": emotion,
            "dialog_emoB": emotion,
            "is_augmented": False,
        })

    return pd.DataFrame(records)


df_ravdess = parse_ravdess(CONFIG["ravdess_path"])
print("RAVDESS:", df_ravdess.shape)
display(df_ravdess.head())
print(df_ravdess["emotion"].value_counts())
print(df_ravdess.groupby(["gender", "emotion"]).size().unstack(fill_value=0))

RAVDESS: (1056, 25)


,filepath,filename,stem,dataset,language,speech_style,corpus_type,emotion_original,emotion,label,...,ravdess_vocal_channel,ravdess_emotion_code,ravdess_intensity,ravdess_statement,ravdess_repetition,ravdess_actor,dialog_context,dialog_emoA,dialog_emoB,is_augmented
0,/content/drive/MyDrive/New Jurnal Cross/datase...,03-01-01-01-01-01-01.wav,03-01-01-01-01-01-01,ravdess,english,monologue,acted_controlled,neutral,neutral,4,...,01,01,01,01,01,01,monologue,neutral,neutral,False
1,/content/drive/MyDrive/New Jurnal Cross/datase...,03-01-01-01-01-02-01.wav,03-01-01-01-01-02-01,ravdess,english,monologue,acted_controlled,neutral,neutral,4,...,01,01,01,01,02,01,monologue,neutral,neutral,False
2,/content/drive/MyDrive/New Jurnal Cross/datase...,03-01-01-01-02-01-01.wav,03-01-01-01-02-01-01,ravdess,english,monologue,acted_controlled,neutral,neutral,4,...,01,01,01,02,01,01,monologue,neutral,neutral,False
3,/content/drive/MyDrive/New Jurnal Cross/datase...,03-01-01-01-02-02-01.wav,03-01-01-01-02-02-01,ravdess,english,monologue,acted_controlled,neutral,neutral,4,...,01,01,01,02,02,01,monologue,neutral,neutral,False
4,/content/drive/MyDrive/New Jurnal Cross/datase...,03-01-03-01-01-01-01.wav,03-01-03-01-01-01-01,ravdess,english,monologue,acted_controlled,happy,happy,3,...,01,03,01,01,01,01,monologue,happy,happy,False


emotion
happy      192
sad        192
fear       192
angry      192
disgust    192
neutral     96
Name: count, dtype: int64
emotion  angry  disgust  fear  happy  neutral  sad
gender                                            
female      96       96    96     96       48   96
male        96       96    96     96       48   96


In [6]:
def parse_resd_filename(stem):
    """
    Expected RESD filename format:
      {speaker_id}_{emoA}_{emoB}_{letter}_{utt_id}

    Possible variant:
      {speaker_id}_{emoA}_{emoB} {letter}_{utt_id}

    Example:
      27_neutral_fear_n_100
      08_sadness_anger a_010
      11_anger_disgust a_040

    Letter mapping:
      a = anger
      d = disgust
      e = enthusiasm  → dibuang
      f = fear
      h = happiness
      n = neutral
      s = sadness
    """
    s = stem.replace(" ", "_")
    parts = s.split("_")

    if len(parts) < 5:
        return None

    speaker_id = parts[0]
    emoA = parts[1].lower()
    emoB = parts[2].lower()
    letter = parts[3].lower()
    utt_id = parts[4]

    valid_dialog_emotions = {
        "anger",
        "disgust",
        "enthusiasm",
        "fear",
        "happiness",
        "neutral",
        "sadness",
    }

    if emoA not in valid_dialog_emotions or emoB not in valid_dialog_emotions:
        return None

    letter_map = {
        "a": "anger",
        "d": "disgust",
        "e": "enthusiasm",
        "f": "fear",
        "h": "happiness",
        "n": "neutral",
        "s": "sadness",
    }

    if letter not in letter_map:
        return None

    letter_emotion_raw = letter_map[letter]
    letter_emotion = RESD_MAP.get(letter_emotion_raw)

    if letter_emotion is None:
        return None

    dialog_pair = tuple(sorted([emoA, emoB]))
    dialog_context = "_".join(dialog_pair)

    return {
        "speaker_id": speaker_id,
        "dialog_context": dialog_context,
        "dialog_emoA": emoA,
        "dialog_emoB": emoB,
        "letter_emotion_raw": letter_emotion_raw,
        "letter_emotion": letter_emotion,
        "utt_id": utt_id,
    }


def parse_resd(base_path):
    """
    RESD bisa memiliki folder train/test, atau langsung berisi folder emosi.
    Label utama diambil dari filename jika berhasil diparse.
    Folder digunakan sebagai fallback dan untuk cek mismatch.
    """
    base = Path(base_path)

    records = []
    parse_failed = 0
    mismatches = 0

    files = find_wav_files(base)

    for f in files:
        stem = f.stem
        parent_name = f.parent.name.lower()

        # Deteksi source split dari path
        path_parts = [p.lower() for p in f.parts]
        if "train" in path_parts:
            source_split = "train"
        elif "test" in path_parts:
            source_split = "test"
        else:
            source_split = "all"

        # Label dari folder sebagai fallback
        folder_emotion = RESD_MAP.get(parent_name)
        if folder_emotion is None:
            for key, value in RESD_MAP.items():
                if key in parent_name:
                    folder_emotion = value
                    break

        parsed = parse_resd_filename(stem)

        if parsed is not None:
            emotion = parsed["letter_emotion"]
            speaker = parsed["speaker_id"]
            dialog_context = parsed["dialog_context"]
            dialog_emoA = parsed["dialog_emoA"]
            dialog_emoB = parsed["dialog_emoB"]
            utt_id = parsed["utt_id"]

            if folder_emotion is not None and folder_emotion != emotion:
                mismatches += 1

        else:
            parse_failed += 1

            if folder_emotion is None:
                continue

            emotion = folder_emotion
            speaker = stem
            dialog_context = "unknown"
            dialog_emoA = folder_emotion
            dialog_emoB = folder_emotion
            utt_id = "unknown"

        if emotion not in LABEL_TO_ID:
            continue

        records.append({
            "filepath": str(f),
            "filename": f.name,
            "stem": stem,
            "dataset": "resd",
            "language": "russian",
            "speech_style": "dialog",
            "corpus_type": "acted_dialog",
            "emotion_original": emotion,
            "emotion": emotion,
            "label": LABEL_TO_ID[emotion],
            "speaker": str(speaker),
            "speaker_original": str(speaker),
            "gender": "unknown",
            "source_split": source_split,
            "dialog_context": dialog_context,
            "dialog_emoA": dialog_emoA,
            "dialog_emoB": dialog_emoB,
            "resd_utt_id": utt_id,
            "is_augmented": False,
        })

    df = pd.DataFrame(records)

    print("RESD parse_failed:", parse_failed)
    print("RESD folder-filename mismatches:", mismatches)

    return df


df_resd = parse_resd(CONFIG["resd_path"])
print("RESD:", df_resd.shape)
display(df_resd.head())
print(df_resd["emotion"].value_counts())
print("Unique speakers:", df_resd["speaker"].nunique())
print("Source split:")
print(df_resd["source_split"].value_counts())

RESD parse_failed: 198
RESD folder-filename mismatches: 0
RESD: (1198, 19)


,filepath,filename,stem,dataset,language,speech_style,corpus_type,emotion_original,emotion,label,speaker,speaker_original,gender,source_split,dialog_context,dialog_emoA,dialog_emoB,resd_utt_id,is_augmented
0,/content/drive/MyDrive/New Jurnal Cross/datase...,01_happiness_anger a_010.wav,01_happiness_anger a_010,resd,russian,dialog,acted_dialog,angry,angry,0,01,01,unknown,test,anger_happiness,happiness,anger,010,False
1,/content/drive/MyDrive/New Jurnal Cross/datase...,01_happiness_anger a_090.wav,01_happiness_anger a_090,resd,russian,dialog,acted_dialog,angry,angry,0,01,01,unknown,test,anger_happiness,happiness,anger,090,False
2,/content/drive/MyDrive/New Jurnal Cross/datase...,01_happiness_anger a_110.wav,01_happiness_anger a_110,resd,russian,dialog,acted_dialog,angry,angry,0,01,01,unknown,test,anger_happiness,happiness,anger,110,False
3,/content/drive/MyDrive/New Jurnal Cross/datase...,02_anger_sadness a_011.wav,02_anger_sadness a_011,resd,russian,dialog,acted_dialog,angry,angry,0,02,02,unknown,test,anger_sadness,anger,sadness,011,False
4,/content/drive/MyDrive/New Jurnal Cross/datase...,02_anger_sadness a_031.wav,02_anger_sadness a_031,resd,russian,dialog,acted_dialog,angry,angry,0,02,02,unknown,test,anger_sadness,anger,sadness,031,False


emotion
fear       223
angry      219
happy      218
neutral    191
disgust    185
sad        162
Name: count, dtype: int64
Unique speakers: 50
Source split:
source_split
train    958
test     240
Name: count, dtype: int64


In [7]:
dfs = []

for name, df in [
    ("emodb", df_emodb),
    ("ravdess", df_ravdess),
    ("resd", df_resd),
]:
    if df is not None and len(df) > 0:
        dfs.append(df)

meta_all = pd.concat(dfs, ignore_index=True)

# Pastikan hanya 6-class
meta_all = meta_all[meta_all["emotion"].isin(CONFIG["unified_emotions"])].reset_index(drop=True)

# Re-assign label agar konsisten
meta_all["label"] = meta_all["emotion"].map(LABEL_TO_ID).astype(int)

# Tambahkan ID unik
meta_all.insert(0, "uid", [f"utt_{i:06d}" for i in range(len(meta_all))])

# Tambahkan path existence check
meta_all["file_exists"] = meta_all["filepath"].apply(lambda x: Path(x).exists())

print("Total metadata:", meta_all.shape)
print("All files exist:", meta_all["file_exists"].all())

display(meta_all.head())

print("\nDistribution per dataset:")
display(meta_all.groupby(["dataset", "emotion"]).size().unstack(fill_value=0))

# Simpan per dataset dan gabungan
meta_all.to_csv(OUT_DIR / "metadata_all_6class.csv", index=False)

for ds in sorted(meta_all["dataset"].unique()):
    df_ds = meta_all[meta_all["dataset"] == ds].reset_index(drop=True)
    df_ds.to_csv(OUT_DIR / f"metadata_{ds}_6class.csv", index=False)
    print(f"Saved metadata_{ds}_6class.csv:", df_ds.shape)

print("Saved metadata_all_6class.csv:", meta_all.shape)

Total metadata: (2972, 28)
All files exist: True


,uid,filepath,filename,stem,dataset,language,speech_style,corpus_type,emotion_original,emotion,...,is_augmented,ravdess_modality,ravdess_vocal_channel,ravdess_emotion_code,ravdess_intensity,ravdess_statement,ravdess_repetition,ravdess_actor,resd_utt_id,file_exists
0,utt_000000,/content/drive/MyDrive/New Jurnal Cross/datase...,03a01Ab.wav,03a01Ab,emodb,german,monologue,acted_controlled,fear,fear,...,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True
1,utt_000001,/content/drive/MyDrive/New Jurnal Cross/datase...,03a01Eb.wav,03a01Eb,emodb,german,monologue,acted_controlled,disgust,disgust,...,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True
2,utt_000002,/content/drive/MyDrive/New Jurnal Cross/datase...,03a01Fa.wav,03a01Fa,emodb,german,monologue,acted_controlled,happy,happy,...,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True
3,utt_000003,/content/drive/MyDrive/New Jurnal Cross/datase...,03a01Nc.wav,03a01Nc,emodb,german,monologue,acted_controlled,neutral,neutral,...,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True
4,utt_000004,/content/drive/MyDrive/New Jurnal Cross/datase...,03a01Tc.wav,03a01Tc,emodb,german,monologue,acted_controlled,sad,sad,...,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True



Distribution per dataset:


emotion,angry,disgust,fear,happy,neutral,sad
dataset,,,,,,
emodb,140,106,123,118,106,125
ravdess,192,192,192,192,96,192
resd,219,185,223,218,191,162


Saved metadata_emodb_6class.csv: (718, 28)
Saved metadata_ravdess_6class.csv: (1056, 28)
Saved metadata_resd_6class.csv: (1198, 28)
Saved metadata_all_6class.csv: (2972, 28)


In [8]:
def make_dataset_audit(meta):
    rows = []

    for ds, df in meta.groupby("dataset"):
        row = {
            "dataset": ds,
            "language": df["language"].iloc[0],
            "speech_style": df["speech_style"].iloc[0],
            "corpus_type": df["corpus_type"].iloc[0],
            "n_files": len(df),
            "n_speakers": df["speaker"].nunique(),
        }

        for emo in CONFIG["unified_emotions"]:
            row[f"n_{emo}"] = int((df["emotion"] == emo).sum())

        rows.append(row)

    audit = pd.DataFrame(rows)
    return audit


audit = make_dataset_audit(meta_all)
display(audit)

audit.to_csv(OUT_DIR / "dataset_audit_6class.csv", index=False)
print("Saved:", OUT_DIR / "dataset_audit_6class.csv")

,dataset,language,speech_style,corpus_type,n_files,n_speakers,n_angry,n_disgust,n_fear,n_happy,n_neutral,n_sad
0,emodb,german,monologue,acted_controlled,718,10,140,106,123,118,106,125
1,ravdess,english,monologue,acted_controlled,1056,24,192,192,192,192,96,192
2,resd,russian,dialog,acted_dialog,1198,50,219,185,223,218,191,162


Saved: /content/drive/MyDrive/New Jurnal Cross/processed_intra_csv/dataset_audit_6class.csv


In [9]:
def speaker_independent_split(
    df,
    train_ratio=0.70,
    val_ratio=0.15,
    test_ratio=0.15,
    seed=42
):
    """
    Split berbasis speaker.
    Semua utterance dari speaker yang sama masuk ke split yang sama.

    Catatan:
    Untuk dataset kecil seperti EmoDB, distribusi label bisa tidak sempurna.
    """
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-6

    speakers = sorted(df["speaker"].unique())
    n_speakers = len(speakers)

    if n_speakers < 3:
        raise ValueError(f"Jumlah speaker terlalu sedikit: {n_speakers}")

    # Split speaker train vs temp
    train_speakers, temp_speakers = train_test_split(
        speakers,
        train_size=train_ratio,
        random_state=seed,
        shuffle=True
    )

    # Split temp menjadi val dan test
    val_fraction_of_temp = val_ratio / (val_ratio + test_ratio)

    val_speakers, test_speakers = train_test_split(
        temp_speakers,
        train_size=val_fraction_of_temp,
        random_state=seed,
        shuffle=True
    )

    train_speakers = set(train_speakers)
    val_speakers = set(val_speakers)
    test_speakers = set(test_speakers)

    df = df.copy()
    df["split"] = "none"

    df.loc[df["speaker"].isin(train_speakers), "split"] = "train"
    df.loc[df["speaker"].isin(val_speakers), "split"] = "val"
    df.loc[df["speaker"].isin(test_speakers), "split"] = "test"

    return df


split_dfs = []

for ds in sorted(meta_all["dataset"].unique()):
    df_ds = meta_all[meta_all["dataset"] == ds].copy().reset_index(drop=True)

    df_split = speaker_independent_split(
        df_ds,
        train_ratio=0.70,
        val_ratio=0.15,
        test_ratio=0.15,
        seed=SEED
    )

    split_dfs.append(df_split)

    out_file = OUT_DIR / f"split_{ds}_6class.csv"
    df_split.to_csv(out_file, index=False)

    print("=" * 70)
    print(ds.upper())
    print("Saved:", out_file)
    print("n files:", len(df_split))
    print("n speakers:", df_split["speaker"].nunique())
    print("\nFiles per split:")
    print(df_split["split"].value_counts())
    print("\nSpeakers per split:")
    print(df_split.groupby("split")["speaker"].nunique())
    print("\nEmotion distribution per split:")
    display(df_split.groupby(["split", "emotion"]).size().unstack(fill_value=0))


meta_intra_split = pd.concat(split_dfs, ignore_index=True)
meta_intra_split.to_csv(OUT_DIR / "metadata_all_6class_with_intra_split.csv", index=False)

print("Saved combined intra split:")
print(OUT_DIR / "metadata_all_6class_with_intra_split.csv")

EMODB
Saved: /content/drive/MyDrive/New Jurnal Cross/processed_intra_csv/split_emodb_6class.csv
n files: 718
n speakers: 10

Files per split:
split
train    508
test     141
val       69
Name: count, dtype: int64

Speakers per split:
split
test     2
train    7
val      1
Name: speaker, dtype: int64

Emotion distribution per split:


emotion,angry,disgust,fear,happy,neutral,sad
split,,,,,,
test,27,20,27,22,21,24
train,98,76,85,84,74,91
val,15,10,11,12,11,10


RAVDESS
Saved: /content/drive/MyDrive/New Jurnal Cross/processed_intra_csv/split_ravdess_6class.csv
n files: 1056
n speakers: 24

Files per split:
split
train    704
val      176
test     176
Name: count, dtype: int64

Speakers per split:
split
test      4
train    16
val       4
Name: speaker, dtype: int64

Emotion distribution per split:


emotion,angry,disgust,fear,happy,neutral,sad
split,,,,,,
test,32,32,32,32,16,32
train,128,128,128,128,64,128
val,32,32,32,32,16,32


RESD
Saved: /content/drive/MyDrive/New Jurnal Cross/processed_intra_csv/split_resd_6class.csv
n files: 1198
n speakers: 50

Files per split:
split
train    892
test     166
val      140
Name: count, dtype: int64

Speakers per split:
split
test      8
train    35
val       7
Name: speaker, dtype: int64

Emotion distribution per split:


emotion,angry,disgust,fear,happy,neutral,sad
split,,,,,,
test,17,32,53,19,21,24
train,202,142,142,155,119,132
val,0,11,28,44,51,6


Saved combined intra split:
/content/drive/MyDrive/New Jurnal Cross/processed_intra_csv/metadata_all_6class_with_intra_split.csv


In [10]:
def check_speaker_leakage(df, dataset_name):
    split_to_speakers = {
        split: set(df[df["split"] == split]["speaker"].unique())
        for split in ["train", "val", "test"]
    }

    leakage = {}

    leakage["train_val"] = split_to_speakers["train"] & split_to_speakers["val"]
    leakage["train_test"] = split_to_speakers["train"] & split_to_speakers["test"]
    leakage["val_test"] = split_to_speakers["val"] & split_to_speakers["test"]

    print("=" * 70)
    print(dataset_name.upper())

    has_leakage = False
    for k, v in leakage.items():
        print(k, ":", len(v), v)
        if len(v) > 0:
            has_leakage = True

    if not has_leakage:
        print("✅ No speaker leakage")
    else:
        print("⚠️ Speaker leakage detected")

    return leakage


for ds in sorted(meta_intra_split["dataset"].unique()):
    check_speaker_leakage(
        meta_intra_split[meta_intra_split["dataset"] == ds],
        ds
    )

EMODB
train_val : 0 set()
train_test : 0 set()
val_test : 0 set()
✅ No speaker leakage
RAVDESS
train_val : 0 set()
train_test : 0 set()
val_test : 0 set()
✅ No speaker leakage
RESD
train_val : 0 set()
train_test : 0 set()
val_test : 0 set()
✅ No speaker leakage


In [11]:
print("Output files:")
for f in sorted(OUT_DIR.glob("*.csv")):
    print("-", f.name)

print("\nFinal distribution:")
display(meta_all.groupby(["dataset", "emotion"]).size().unstack(fill_value=0))

print("\nIntra split distribution:")
display(meta_intra_split.groupby(["dataset", "split", "emotion"]).size().unstack(fill_value=0))

Output files:
- dataset_audit_6class.csv
- metadata_all_6class.csv
- metadata_all_6class_with_intra_split.csv
- metadata_all_6class_with_intra_split_optimized.csv
- metadata_emodb_6class.csv
- metadata_ravdess_6class.csv
- metadata_resd_6class.csv
- split_emodb_6class.csv
- split_emodb_6class_optimized.csv
- split_ravdess_6class.csv
- split_ravdess_6class_optimized.csv
- split_resd_6class.csv
- split_resd_6class_optimized.csv

Final distribution:


emotion,angry,disgust,fear,happy,neutral,sad
dataset,,,,,,
emodb,140,106,123,118,106,125
ravdess,192,192,192,192,96,192
resd,219,185,223,218,191,162



Intra split distribution:


emotion        angry  disgust  fear  happy  neutral  sad
dataset split                                           
emodb   test      27       20    27     22       21   24
        train     98       76    85     84       74   91
        val       15       10    11     12       11   10
ravdess test      32       32    32     32       16   32
        train    128      128   128    128       64  128
        val       32       32    32     32       16   32
resd    test      17       32    53     19       21   24
        train    202      142   142    155      119  132
        val        0       11    28     44       51    6

In [12]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

def distribution_score(df_split, emotions):
    """
    Semakin kecil skor, semakin baik.
    Penalti besar jika ada kelas kosong di val/test.
    """
    global_dist = df_split["emotion"].value_counts(normalize=True).reindex(emotions, fill_value=0)

    score = 0.0
    penalty = 0.0

    for split in ["train", "val", "test"]:
        d = df_split[df_split["split"] == split]
        split_dist = d["emotion"].value_counts(normalize=True).reindex(emotions, fill_value=0)

        # Jarak distribusi ke distribusi global
        score += np.abs(split_dist.values - global_dist.values).sum()

        # Penalti kelas kosong
        counts = d["emotion"].value_counts().reindex(emotions, fill_value=0)

        if split in ["val", "test"]:
            missing = (counts == 0).sum()
            penalty += missing * 1000

        if split == "train":
            missing = (counts == 0).sum()
            penalty += missing * 2000

    return score + penalty


def optimized_speaker_independent_split(
    df,
    emotions,
    train_ratio=0.70,
    val_ratio=0.15,
    test_ratio=0.15,
    n_trials=5000,
    seed=42,
    min_per_class_val=1,
    min_per_class_test=1,
):
    """
    Mencari speaker-independent split terbaik dengan random search.
    Tidak menjamin sempurna, tapi biasanya cukup untuk dataset kecil/menengah.
    """
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-6

    speakers = sorted(df["speaker"].unique())
    rng = np.random.default_rng(seed)

    best_df = None
    best_score = float("inf")
    best_seed = None

    for trial in range(n_trials):
        s = int(rng.integers(0, 10_000_000))

        try:
            train_speakers, temp_speakers = train_test_split(
                speakers,
                train_size=train_ratio,
                random_state=s,
                shuffle=True
            )

            val_fraction_of_temp = val_ratio / (val_ratio + test_ratio)

            val_speakers, test_speakers = train_test_split(
                temp_speakers,
                train_size=val_fraction_of_temp,
                random_state=s,
                shuffle=True
            )

            train_speakers = set(train_speakers)
            val_speakers = set(val_speakers)
            test_speakers = set(test_speakers)

            tmp = df.copy()
            tmp["split"] = "none"
            tmp.loc[tmp["speaker"].isin(train_speakers), "split"] = "train"
            tmp.loc[tmp["speaker"].isin(val_speakers), "split"] = "val"
            tmp.loc[tmp["speaker"].isin(test_speakers), "split"] = "test"

            # Hard constraint: val dan test harus punya semua kelas
            val_counts = tmp[tmp["split"] == "val"]["emotion"].value_counts().reindex(emotions, fill_value=0)
            test_counts = tmp[tmp["split"] == "test"]["emotion"].value_counts().reindex(emotions, fill_value=0)
            train_counts = tmp[tmp["split"] == "train"]["emotion"].value_counts().reindex(emotions, fill_value=0)

            if (train_counts == 0).any():
                continue
            if (val_counts < min_per_class_val).any():
                continue
            if (test_counts < min_per_class_test).any():
                continue

            score = distribution_score(tmp, emotions)

            if score < best_score:
                best_score = score
                best_df = tmp
                best_seed = s

        except Exception:
            continue

    if best_df is None:
        raise RuntimeError(
            "Tidak menemukan split valid. Coba naikkan n_trials atau turunkan constraint."
        )

    return best_df, best_seed, best_score

In [13]:
EMOTIONS = CONFIG["unified_emotions"]

optimized_split_dfs = []

for ds in sorted(meta_all["dataset"].unique()):
    df_ds = meta_all[meta_all["dataset"] == ds].copy().reset_index(drop=True)

    df_split, best_seed, best_score = optimized_speaker_independent_split(
        df_ds,
        emotions=EMOTIONS,
        train_ratio=0.70,
        val_ratio=0.15,
        test_ratio=0.15,
        n_trials=10000,
        seed=SEED,
        min_per_class_val=1,
        min_per_class_test=1,
    )

    optimized_split_dfs.append(df_split)

    out_file = OUT_DIR / f"split_{ds}_6class_optimized.csv"
    df_split.to_csv(out_file, index=False)

    print("=" * 80)
    print(ds.upper())
    print("Best seed :", best_seed)
    print("Best score:", best_score)
    print("Saved     :", out_file)
    print("\nFiles per split:")
    print(df_split["split"].value_counts())
    print("\nSpeakers per split:")
    print(df_split.groupby("split")["speaker"].nunique())
    print("\nEmotion distribution per split:")
    display(df_split.groupby(["split", "emotion"]).size().unstack(fill_value=0))


meta_intra_split_opt = pd.concat(optimized_split_dfs, ignore_index=True)
meta_intra_split_opt.to_csv(
    OUT_DIR / "metadata_all_6class_with_intra_split_optimized.csv",
    index=False
)

print("Saved combined optimized split:")
print(OUT_DIR / "metadata_all_6class_with_intra_split_optimized.csv")

EMODB
Best seed : 5717280
Best score: 0.06796490684527559
Saved     : /content/drive/MyDrive/New Jurnal Cross/processed_intra_csv/split_emodb_6class_optimized.csv

Files per split:
split
train    498
test     149
val       71
Name: count, dtype: int64

Speakers per split:
split
test     2
train    7
val      1
Name: speaker, dtype: int64

Emotion distribution per split:


emotion,angry,disgust,fear,happy,neutral,sad
split,,,,,,
test,29,22,25,24,22,27
train,98,73,85,82,74,86
val,13,11,13,12,10,12


RAVDESS
Best seed : 892509
Best score: 0.0
Saved     : /content/drive/MyDrive/New Jurnal Cross/processed_intra_csv/split_ravdess_6class_optimized.csv

Files per split:
split
train    704
test     176
val      176
Name: count, dtype: int64

Speakers per split:
split
test      4
train    16
val       4
Name: speaker, dtype: int64

Emotion distribution per split:


emotion,angry,disgust,fear,happy,neutral,sad
split,,,,,,
test,32,32,32,32,16,32
train,128,128,128,128,64,128
val,32,32,32,32,16,32


RESD
Best seed : 9668019
Best score: 0.2769031356486025
Saved     : /content/drive/MyDrive/New Jurnal Cross/processed_intra_csv/split_resd_6class_optimized.csv

Files per split:
split
train    873
val      186
test     139
Name: count, dtype: int64

Speakers per split:
split
test      8
train    35
val       7
Name: speaker, dtype: int64

Emotion distribution per split:


emotion,angry,disgust,fear,happy,neutral,sad
split,,,,,,
test,34,15,29,23,20,18
train,154,135,162,160,140,122
val,31,35,32,35,31,22


Saved combined optimized split:
/content/drive/MyDrive/New Jurnal Cross/processed_intra_csv/metadata_all_6class_with_intra_split_optimized.csv


Keputusan

Gunakan file ini sebagai metadata utama intra-corpus:

/content/drive/MyDrive/New Jurnal Cross/processed_intra_csv/metadata_all_6class_with_intra_split_optimized.csv

Dan per dataset:

split_emodb_6class_optimized.csv
split_ravdess_6class_optimized.csv
split_resd_6class_optimized.csv

| Dataset |                   Train |                     Val |                    Test | Catatan                       |
| ------- | ----------------------: | ----------------------: | ----------------------: | ----------------------------- |
| EmoDB   |  498 / 718 = **69.36%** |    71 / 718 = **9.89%** |  149 / 718 = **20.75%** | karena hanya 10 speaker       |
| RAVDESS | 704 / 1056 = **66.67%** | 176 / 1056 = **16.67%** | 176 / 1056 = **16.67%** | sangat seimbang               |
| RESD    | 873 / 1198 = **72.87%** | 186 / 1198 = **15.53%** | 139 / 1198 = **11.60%** | hasil optimized speaker split |
